# Role-Design & Delegation — Practical Notebook

This practical turns multi-agent design ideas into artifacts that can be implemented later:

- role cards
- RACI ownership
- handoff contracts
- incentive-risk controls
- loop controls
- a build-ready research-bot blueprint

The code is framework-neutral. That is deliberate: strong team design should come before framework syntax.


## What you will build

You will inspect and validate a blueprint for an evidence-backed research bot.

The team design includes:

- **Editor Orchestrator** — owns final answer quality
- **Planner** — turns the user question into research subquestions
- **Researcher** — gathers evidence notes
- **Writer** — drafts from evidence only
- **Fact-Checker** — verifies claims against evidence
- **Memory Manager** — stores only verified reusable context

The next build session can collapse or expand these roles depending on implementation cost.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
from pathlib import Path
from pprint import pprint
import json

from role_models import AgentTeamBlueprint, RoleCard, RACIEntry, HandoffContract
from validators import (
    load_blueprint,
    validate_blueprint,
    validate_role_cards,
    validate_raci,
    validate_handoff_contracts,
    validate_incentives,
    validate_loop_controls,
    render_report,
    score_blueprint,
)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

sample_path = DATA_DIR / "sample_research_bot_blueprint.json"
flawed_path = DATA_DIR / "flawed_research_bot_blueprint.json"

print(f"Using data folder: {DATA_DIR}")
print(f"Sample blueprint exists: {sample_path.exists()}")


Using data folder: e:\BIA\BIA_GenAI_May_26\role_design_delegation\role_design_delegation\data
Sample blueprint exists: True


## 1. Load the research-bot blueprint

A blueprint is a design document that can later become implementation:

- roles become agents or functions
- RACI becomes routing and ownership logic
- handoff contracts become structured messages
- loop controls become termination and retry rules


In [4]:
blueprint = load_blueprint(sample_path)

print("Project:", blueprint.project_name)
print("Delegation pattern:", blueprint.delegation_pattern.value)
print("User goal:", blueprint.user_goal)
print("\nRoles:")
for role_name in blueprint.role_names:
    print(" -", role_name)


Project: Evidence-Backed Research Bot
Delegation pattern: hybrid
User goal: Answer a business question with a concise response supported by source-backed claims.

Roles:
 - Editor Orchestrator
 - Planner
 - Researcher
 - Writer
 - Fact-Checker
 - Memory Manager


## 2. Inspect role cards

A role card answers:

- What is this role responsible for?
- What artifact does it own?
- What decisions can it make?
- When should it stop?
- When should it escalate?

Without these boundaries, agent teams often duplicate work, fight over decisions, or loop forever.


In [6]:
def print_role_summary(role_name: str) -> None:
    role = blueprint.get_role(role_name)
    if role is None:
        print(f"Role not found: {role_name}")
        return

    print(f"Role: {role.name}")
    print(f"Type: {role.role_type.value}")
    print(f"Mission: {role.mission}\n")

    print("Outputs owned:")
    for item in role.outputs_owned:
        print(" -", item)

    print("\nDecision rights:")
    for item in role.decision_rights:
        print(" -", item)

    print("\nStop conditions:")
    for item in role.stop_conditions:
        print(" -", item)

print_role_summary("Fact-Checker")


Role: Fact-Checker
Type: Fact-Checker
Mission: Verify that every important claim in the draft is supported by the evidence table.

Outputs owned:
 - Fact-check report

Decision rights:
 - Can block final approval when major claims are unsupported
 - Can request revisions from Writer
 - Cannot rewrite the final answer without Editor approval

Stop conditions:
 - All major claims are checked
 - Unsupported claims are listed with fixes
 - Maximum revision cycles reached


### Try this

Change the role name in the previous cell to:

- `Editor Orchestrator`
- `Researcher`
- `Writer`
- `Memory Manager`

Discuss: which role is allowed to block final approval?


In [7]:
blueprint.roles[0].outputs_owned

['Final answer', 'Revision decision']

In [ ]:
role_findings = validate_role_cards(blueprint)
print(render_report(role_findings))


Role-Design Blueprint Validation Report
Score: 100/100
Status: build-ready
Errors: 0 | Warnings: 0 | Info: 0

No findings. The blueprint is ready to convert into implementation tasks.


## 3. Inspect RACI ownership

RACI is a way to debug ownership before coding.

- **Responsible** — does the work
- **Accountable** — signs off; exactly one per task
- **Consulted** — gives input before or during the task
- **Informed** — receives updates after the task

A common multi-agent failure is giving multiple agents responsibility without assigning final accountability.


In [8]:
for row in blueprint.raci:
    print(f"Task: {row.task}")
    print(f"  Responsible: {', '.join(row.responsible)}")
    print(f"  Accountable: {row.accountable}")
    print(f"  Artifact: {row.artifact}")
    print(f"  Done when: {'; '.join(row.acceptance_criteria)}")
    print()


Task: Interpret user question
  Responsible: Editor Orchestrator
  Accountable: Editor Orchestrator
  Artifact: Clarified user goal
  Done when: Goal is specific enough for research; Output format is known; Ambiguities are stated

Task: Create research plan
  Responsible: Planner
  Accountable: Editor Orchestrator
  Artifact: Research plan
  Done when: 2-4 subquestions are defined; Evidence needs are testable; Plan fits the available time

Task: Gather evidence
  Responsible: Researcher
  Accountable: Planner
  Artifact: Evidence table
  Done when: Each note maps to a subquestion; Each factual note has a source label; Evidence gaps are listed

Task: Draft answer
  Responsible: Writer
  Accountable: Editor Orchestrator
  Artifact: Draft answer
  Done when: Draft directly answers the question; Draft uses only evidence-table claims; Uncertainty is stated

Task: Verify claims
  Responsible: Fact-Checker
  Accountable: Editor Orchestrator
  Artifact: Fact-check report
  Done when: Major cla

In [8]:
raci_findings = validate_raci(blueprint)
print(render_report(raci_findings))


Role-Design Blueprint Validation Report
Score: 100/100
Status: build-ready
Errors: 0 | Warnings: 0 | Info: 0

No findings. The blueprint is ready to convert into implementation tasks.


### RACI discussion

Look at the task **Gather evidence**.

- Why is Researcher Responsible?
- Why is Planner Accountable?
- Should Editor Orchestrator be Accountable instead?
- What would change in a smaller four-agent implementation?


## 4. Inspect handoff contracts

A handoff contract is the API between roles.

Bad handoff:

> Researcher sends notes to Writer.

Better handoff:

> Researcher sends claims, source labels, confidence, and evidence gaps.

The second version can be validated and converted into structured messages later.


In [10]:
for contract in blueprint.handoff_contracts:
    print(f"Contract: {contract.name}")
    print(f"  {contract.source_role} → {contract.target_role}")
    print(f"  Trigger: {contract.trigger}")
    print(f"  Output fields: {', '.join(contract.output_schema.keys())}")
    print(f"  Retry: {contract.retry_policy}")
    print()


Contract: Planner to Researcher: research brief
  Planner → Researcher
  Trigger: Editor approves clarified user goal.
  Output fields: research_questions, evidence_needs, acceptance_criteria
  Retry: Planner may revise once if Researcher flags ambiguity; then escalate to Editor Orchestrator.

Contract: Researcher to Writer: evidence table
  Researcher → Writer
  Trigger: Researcher reaches minimum evidence threshold or documents evidence gaps.
  Output fields: claims, source_labels, confidence, evidence_gaps
  Retry: Researcher gets one additional evidence pass if Writer finds missing support.

Contract: Writer to Fact-Checker: draft for verification
  Writer → Fact-Checker
  Trigger: Writer completes first draft from evidence table.
  Output fields: draft_answer, claim_map, known_limitations
  Retry: Writer revises at most twice based on Fact-Checker report.

Contract: Fact-Checker to Editor: verification report
  Fact-Checker → Editor Orchestrator
  Trigger: Fact-Checker completes c

In [11]:
handoff_findings = validate_handoff_contracts(blueprint)
print(render_report(handoff_findings))


Role-Design Blueprint Validation Report
Score: 100/100
Status: build-ready
Errors: 0 | Warnings: 0 | Info: 0

No findings. The blueprint is ready to convert into implementation tasks.


### Contract deep dive

Run this cell to inspect the `Researcher → Writer` contract. This is the most important handoff in a research bot because it controls whether the Writer can invent unsupported claims.


In [12]:
researcher_writer = next(
    contract for contract in blueprint.handoff_contracts
    if contract.source_role == "Researcher" and contract.target_role == "Writer"
)

pprint(researcher_writer.model_dump())


{'escalation_path': 'Planner resolves research scope; Editor Orchestrator '
                    'resolves business trade-offs.',
 'input_schema': {'evidence_needs': 'Required evidence types',
                  'research_questions': 'Subquestions from the research plan'},
 'logging_fields': ['claim_count',
                    'source_count',
                    'low_confidence_count',
                    'evidence_gap_count'],
 'name': 'Researcher to Writer: evidence table',
 'output_schema': {'claims': 'Atomic factual claims',
                   'confidence': 'High, medium, or low confidence for each '
                                 'claim',
                   'evidence_gaps': 'Known missing or weak evidence',
                   'source_labels': 'Source IDs, document IDs, or retrieval '
                                    'references'},
 'quality_checks': ['Each claim has a source label',
                    'Confidence is present for each claim',
                    'Evidence gaps a

## 5. Incentive alignment scan

Agents optimize for what their prompts and evaluation criteria reward.

Examples:

- Reward speed only → shallow research
- Reward polish only → unsupported claims
- Reward number of issues found → endless critique
- Reward low cost only → skipped verification

The blueprint must balance usefulness, evidence, safety, latency, and cost.


In [13]:
policy = blueprint.incentive_policy

print("Primary metric:")
print("-", policy.primary_success_metric)

print("\nBalancing metrics:")
for metric in policy.balancing_metrics:
    print("-", metric)

print("\nAnti-goals:")
for anti_goal in policy.anti_goals:
    print("-", anti_goal)

print("\nMitigation rules:")
for rule in policy.mitigation_rules:
    print("-", rule)


Primary metric:
- Useful answer with verified, evidence-backed claims

Balancing metrics:
- Conciseness
- Latency within classroom demo limits
- Cost awareness
- Coverage of the user's actual decision need
- Safe handling of uncertain or sensitive information

Anti-goals:
- Do not invent unsupported claims
- Do not optimize for citation count over citation quality
- Do not hide uncertainty to sound confident
- Do not store unverified claims in long-term memory

Mitigation rules:
- Unsupported claims must be removed or marked uncertain
- Evidence gaps are allowed when stated transparently
- Fact-Checker can request at most two revision cycles
- Final answer must include limitations when evidence is weak


In [14]:
incentive_findings = validate_incentives(blueprint)
print(render_report(incentive_findings))


Role-Design Blueprint Validation Report
Score: 100/100
Status: build-ready
Errors: 0 | Warnings: 0 | Info: 0

No findings. The blueprint is ready to convert into implementation tasks.


## 6. Loop controls

A multi-agent system needs explicit stop rules.

Look for:

- maximum team iterations
- maximum revision cycles
- escalation after repeated failures
- human escalation rule
- termination rule

Without these, a Writer and Critic can revise forever.


In [15]:
loop = blueprint.loop_control

print("Max team iterations:", loop.max_team_iterations)
print("Max revision cycles:", loop.max_revision_cycles)
print("Escalation after failures:", loop.escalation_after_failures)
print("Human escalation rule:", loop.human_escalation_rule)
print("Termination rule:", loop.termination_rule)

loop_findings = validate_loop_controls(blueprint)
print("\n" + render_report(loop_findings))


Max team iterations: 3
Max revision cycles: 2
Escalation after failures: 2
Human escalation rule: Ask for human review when evidence is insufficient after two revision cycles or the user request is high-stakes.
Termination rule: Stop when final answer is approved, max revision cycles are reached, or human escalation is triggered.

Role-Design Blueprint Validation Report
Score: 100/100
Status: build-ready
Errors: 0 | Warnings: 0 | Info: 0

No findings. The blueprint is ready to convert into implementation tasks.


## 7. Run the full blueprint report

The full report combines:

- role-card checks
- RACI checks
- handoff-contract checks
- incentive checks
- loop-control checks

A strong design does not need to be perfect. It needs to make implementation risks visible before code is written.


In [16]:
findings = validate_blueprint(blueprint)
score = score_blueprint(findings)

print(render_report(findings))


Role-Design Blueprint Validation Report
Score: 100/100
Status: build-ready
Errors: 0 | Warnings: 0 | Info: 0

No findings. The blueprint is ready to convert into implementation tasks.


In [17]:
# Save the report for trainer review or group submission.
output_path = PROJECT_ROOT / "outputs" / "sample_blueprint_report.txt"
output_path.parent.mkdir(exist_ok=True)

output_path.write_text(render_report(findings), encoding="utf-8")
print(f"Saved report to: {output_path}")


Saved report to: E:\BIA\BIA_GenAI_May_26\role_design_delegation\role_design_delegation\outputs\sample_blueprint_report.txt


## 8. Debug an intentionally flawed blueprint

Before running the next cell, predict at least three problems in the flawed design.

Look for:

- missing Accountable owners
- unknown role names
- vague or missing handoff schemas
- overlapping ownership
- speed/cost-only incentives
- missing stop rules


In [18]:
flawed = load_blueprint(flawed_path)
flawed_findings = validate_blueprint(flawed)

print(render_report(flawed_findings))


Role-Design Blueprint Validation Report
Score: 0/100
Status: needs design revision
Errors: 7 | Warnings: 26 | Info: 0

1. [WARNING] Role cards
   Finding: Researcher has no decision rights.
   Fix: Add 2-4 concrete decision rights so the role is implementable.

2. [WARNING] Role cards
   Finding: Researcher has no failure modes.
   Fix: Add 2-4 concrete failure modes so the role is implementable.

3. [WARNING] Role cards
   Finding: Researcher has no stop conditions.
   Fix: Add 2-4 concrete stop conditions so the role is implementable.

4. [WARNING] Role cards
   Finding: Writer has no failure modes.
   Fix: Add 2-4 concrete failure modes so the role is implementable.

5. [WARNING] Role cards
   Finding: Writer has no stop conditions.
   Fix: Add 2-4 concrete stop conditions so the role is implementable.

6. [WARNING] Role cards
   Finding: Critic has no outputs owned.
   Fix: Add 2-4 concrete outputs owned so the role is implementable.

7. [WARNING] Role cards
   Finding: Critic has 

### Group exercise

Fix the flawed blueprint in one of these ways:

1. Add a proper Editor Orchestrator role.
2. Add a final approval RACI row.
3. Rewrite the Researcher → Writer handoff with required fields.
4. Add stop conditions for Critic.
5. Replace speed/cost-only incentives with evidence and safety metrics.

Save your edited file under a new name and run the validator again.


In [19]:
# Example: copy the flawed file to an editable version.
editable_path = DATA_DIR / "my_group_blueprint.json"

if not editable_path.exists():
    editable_path.write_text(flawed_path.read_text(encoding="utf-8"), encoding="utf-8")
    print(f"Created editable copy: {editable_path}")
else:
    print(f"Editable copy already exists: {editable_path}")

print("Edit this JSON file, then run the next cell.")


Created editable copy: E:\BIA\BIA_GenAI_May_26\role_design_delegation\role_design_delegation\data\my_group_blueprint.json
Edit this JSON file, then run the next cell.


In [20]:
# Run this after editing data/my_group_blueprint.json
try:
    group_blueprint = load_blueprint(editable_path)
    group_findings = validate_blueprint(group_blueprint)
    print(render_report(group_findings))
except Exception as exc:
    print("Could not validate the edited blueprint yet.")
    print("Check that the JSON syntax is valid and all required fields exist.")
    print("Error:", exc)


Role-Design Blueprint Validation Report
Score: 0/100
Status: needs design revision
Errors: 7 | Warnings: 26 | Info: 0

1. [WARNING] Role cards
   Finding: Researcher has no decision rights.
   Fix: Add 2-4 concrete decision rights so the role is implementable.

2. [WARNING] Role cards
   Finding: Researcher has no failure modes.
   Fix: Add 2-4 concrete failure modes so the role is implementable.

3. [WARNING] Role cards
   Finding: Researcher has no stop conditions.
   Fix: Add 2-4 concrete stop conditions so the role is implementable.

4. [WARNING] Role cards
   Finding: Writer has no failure modes.
   Fix: Add 2-4 concrete failure modes so the role is implementable.

5. [WARNING] Role cards
   Finding: Writer has no stop conditions.
   Fix: Add 2-4 concrete stop conditions so the role is implementable.

6. [WARNING] Role cards
   Finding: Critic has no outputs owned.
   Fix: Add 2-4 concrete outputs owned so the role is implementable.

7. [WARNING] Role cards
   Finding: Critic has 

## 9. Optional LLM critique

This section is optional. It requires `OPENAI_API_KEY` in `.env`.

Use it to compare deterministic validator feedback with qualitative architectural feedback from an LLM.

The deterministic validator is better at rule checks. The LLM critique is better at explaining design trade-offs in natural language.


In [21]:
from llm_reviewer import critique_blueprint_with_llm

# COST NOTE: Optional. With gpt-4o-mini and the sample blueprint, this is expected
# to cost only a few cents or less depending on current pricing and token count.
critique = critique_blueprint_with_llm(sample_path)
print(critique)


**Feedback:**

1. **Role Clarity:**
   - Overall roles are well-defined, but ensure clarity in boundaries between similar tasks of different roles, especially between "Researcher" and "Fact-Checker."

2. **RACI Ownership:**
   - Clearly delineated, though the consistent accountability of the "Editor Orchestrator" suggests possible bottlenecks. Consider distributing accountability to enhance efficiency.

3. **Handoff Contract Completeness:**
   - Contracts between roles are adequately covered, but ensure that all roles understand the sequence and criteria for these handoffs to prevent delays or miscommunications.

4. **Incentive Alignment:**
   - Roles seem motivated by shared success metrics. Ensure that metrics don't encourage excessive conservatism from "Fact-Checker" or "Memory Manager" regarding what is stored.

5. **Conflict and Loop Risks:**
   - Some potential for looping between "Writer" and "Fact-Checker" exists. Consider tightening revision criteria or involving the "Editor O

## 10. Practical wrap-up

You now have:

- a role-card structure
- a RACI ownership model
- handoff contracts
- incentive-risk controls
- loop controls
- a blueprint that can become a multi-agent research bot

In the next build, these design artifacts become implementation:

- role cards → agent system prompts
- RACI → routing and approval logic
- handoff contracts → structured JSON messages
- loop controls → retry, escalation, and termination logic
